<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V22b_Feature_Ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V22b — Multi-City AD mit Feature-Set-Ablation (Baseline / Weather / Geo)

**Zweck:** Systematischer Vergleich dreier Feature-Sets auf dem Multi-City Oracle + MOMENT Loop.

| Feature-Set | Zusätzliche Features | Motivation |
|---|---|---|
| `baseline` | — (9 Features: demand + zyklisch + weekend) | Referenz, identisch zu V22 |
| `weather` | +temperature, +humidity, +precipitation, +wind_speed | Exogener Kontext: Wetter beeinflusst Nachfrage |
| `geo` | +latitude | Räumlicher Kontext: Breitengrad als Proxy für Stadtlage |

**Scoring** bleibt immer auf `n_lends` + `n_returns` — Extra-Features geben dem Modell nur zusätzlichen Kontext.  
**MOMENT** läuft nur 1× mit Baseline (nutzt nur Demand-Channels univariat).

Basiert vollständig auf V22-Logik. Jedes Feature-Set bekommt eigene Modelle, Sequenzen, Ergebnisse.

In [1]:
# ══════════════════════════════════════════════════════════════
# 0a — Colab Setup
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ══════════════════════════════════════════════════════════════
# 0b — Imports
# ══════════════════════════════════════════════════════════════
import os, math, json, random, warnings, time, re, glob, copy, pickle
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import poisson
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, precision_recall_curve,
    classification_report
)
from numpy.lib.stride_tricks import sliding_window_view

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
print(f"TF {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

TF 2.19.0, GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ══════════════════════════════════════════════════════════════
# 0c — Config (V22b: mit Feature-Sets + Wetter-Pfad)
# ══════════════════════════════════════════════════════════════
DATA_BASE    = '/content/drive/MyDrive/BA_Colab/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

RUN_NAME    = 'v22b_feature_ablation'
RESULTS_DIR = f'{DATA_BASE}/{RUN_NAME}'
MODELS_DIR  = f'{RESULTS_DIR}/models'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Paths
geo_path           = f'{CLEANED_BASE}/geo_information/geo_information.parquet'
station_names_path = f'{DATA_BASE}/station_names/station_names.parquet'
weather_path       = f'{DATA_BASE}/weather/weather.parquet'   # ← NEU

# City-specific demand paths (wird unten per Discovery erweitert)
CITY_DEMAND = {
    "Mannheim":   f'{CLEANED_BASE}/demand/Mannheim/demand_cleaned.parquet',
    "Heidelberg": f'{CLEANED_BASE}/demand/Heidelberg/demand_cleaned.parquet',
}

@dataclass
class V17bConfig:
    aggregation_minutes: int = 60
    train_ratio: float = 0.67
    val_ratio: float = 0.82
    min_events_per_day: float = 3.0
    rolling_days: int = 56
    min_context_obs: int = 20

    # Model architecture
    ae_window_size: int = 24
    ae_latent_dim: int = 32
    ae_layers: int = 2
    ae_dropout: float = 0.10
    ae_epochs: int = 50
    ae_batch_size: int = 2048
    ae_lr: float = 1e-3
    ae_early_stop: int = 8
    eval_batch_size: int = 2048

    # Regime thresholds
    low_demand_q: float = 0.33
    high_demand_q: float = 0.67

    require_contiguous: bool = True
    use_bidirectional: bool = False

    # Baseline features (Standard V22)
    ae_features: List[str] = field(default_factory=lambda: [
        "n_lends", "n_returns",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos",
        "month_sin", "month_cos", "is_weekend"
    ])
    ae_score_features: List[str] = field(default_factory=lambda: ["n_lends", "n_returns"])

    # Injection
    injection_rate: float = 0.015
    injection_seed: int = 42

    # Fine-tune
    finetune_ratio: float = 0.05

cfg = V17bConfig()

# ── Feature-Sets ──────────────────────────────────────────────
FEATURE_SETS = {
    "baseline": {
        "ae_features": list(cfg.ae_features),  # 9 Features
        "ae_score_features": list(cfg.ae_score_features),
    },
    "weather": {
        "ae_features": list(cfg.ae_features) + [
            "temperature", "humidity", "precipitation", "wind_speed"
        ],  # 13 Features
        "ae_score_features": list(cfg.ae_score_features),
    },
    "geo": {
        "ae_features": list(cfg.ae_features) + ["latitude"],  # 10 Features
        "ae_score_features": list(cfg.ae_score_features),
    },
}

print("Feature-Sets:")
for name, fs in FEATURE_SETS.items():
    print(f"  {name:12s}: {len(fs['ae_features'])} Features → {fs['ae_features']}")
print(f"\nResults: {RESULTS_DIR}")

Feature-Sets:
  baseline    : 9 Features → ['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']
  weather     : 13 Features → ['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'temperature', 'humidity', 'precipitation', 'wind_speed']
  geo         : 10 Features → ['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'latitude']

Results: /content/drive/MyDrive/BA_Colab/data/v22b_feature_ablation


---
## Shared Functions (aus V22/V17 übernommen)

In [25]:
# ══════════════════════════════════════════════════════════════
# 1 — Aggregation
# ══════════════════════════════════════════════════════════════
def aggregate_station_level(df: pd.DataFrame, minutes: int = 60,
                            add_relative: bool = False) -> pd.DataFrame:
    out = df.copy()
    freq = f"{minutes}min"
    out["time_bin"] = out["timestamp"].dt.floor(freq)

    agg = (
        out.groupby(
            ["station_id", "station_name_id", "station_name", "location_id", "time_bin"],
            as_index=False
        )
        .agg({
            "n_lends": "sum",
            "n_returns": "sum",
            "latitude": "first",
            "longitude": "first"
        })
        .rename(columns={"time_bin": "hour_ts"})
    )

    agg["total_demand"] = agg["n_lends"] + agg["n_returns"]
    agg["net_flow"] = agg["n_returns"] - agg["n_lends"]
    agg["abs_net_flow"] = agg["net_flow"].abs()

    agg["hour"] = agg["hour_ts"].dt.hour
    agg["dow"] = agg["hour_ts"].dt.dayofweek
    agg["month"] = agg["hour_ts"].dt.month
    agg["is_weekend"] = agg["dow"].isin([5, 6]).astype(int)

    agg["hour_sin"] = np.sin(2 * np.pi * agg["hour"] / 24)
    agg["hour_cos"] = np.cos(2 * np.pi * agg["hour"] / 24)
    agg["dow_sin"]  = np.sin(2 * np.pi * agg["dow"] / 7)
    agg["dow_cos"]  = np.cos(2 * np.pi * agg["dow"] / 7)
    agg["month_sin"] = np.sin(2 * np.pi * (agg["month"] - 1) / 12)
    agg["month_cos"] = np.cos(2 * np.pi * (agg["month"] - 1) / 12)

    agg = agg.sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    return agg

In [26]:
# ══════════════════════════════════════════════════════════════
# 2 — Gap-Fill
# ══════════════════════════════════════════════════════════════
def fill_missing_time_bins(x: pd.DataFrame, minutes: int = 60) -> pd.DataFrame:
    freq = f"{minutes}min"
    parts = []
    x = x.copy().sort_values(["station_id", "hour_ts"])
    x = (
        x.groupby(["station_id", "hour_ts"], as_index=False)
         .agg({
             "station_name_id": "first", "station_name": "first",
             "location_id": "first", "latitude": "first",
             "longitude": "first", "n_lends": "sum", "n_returns": "sum",
         })
    )
    key_cols = ["station_id", "station_name_id", "station_name",
                "location_id", "latitude", "longitude"]

    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").copy()
        full_idx = pd.date_range(g["hour_ts"].min(), g["hour_ts"].max(), freq=freq)
        g = g.set_index("hour_ts").reindex(full_idx)
        g.index.name = "hour_ts"
        g["n_lends"]   = g["n_lends"].fillna(0).astype(int)
        g["n_returns"] = g["n_returns"].fillna(0).astype(int)
        for col in key_cols:
            g[col] = g[col].ffill().bfill()
        parts.append(g.reset_index())

    result = pd.concat(parts, ignore_index=True)
    result["total_demand"] = result["n_lends"] + result["n_returns"]
    result["net_flow"]     = result["n_returns"] - result["n_lends"]
    result["abs_net_flow"] = result["net_flow"].abs()

    result["hour"] = result["hour_ts"].dt.hour
    result["dow"]  = result["hour_ts"].dt.dayofweek
    result["month"] = result["hour_ts"].dt.month
    result["is_weekend"] = result["dow"].isin([5, 6]).astype(int)

    result["hour_sin"]  = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"]  = np.cos(2 * np.pi * result["hour"] / 24)
    result["dow_sin"]   = np.sin(2 * np.pi * result["dow"] / 7)
    result["dow_cos"]   = np.cos(2 * np.pi * result["dow"] / 7)
    result["month_sin"] = np.sin(2 * np.pi * (result["month"] - 1) / 12)
    result["month_cos"] = np.cos(2 * np.pi * (result["month"] - 1) / 12)

    return result.sort_values(["station_id", "hour_ts"]).reset_index(drop=True)

In [27]:
# ══════════════════════════════════════════════════════════════
# 3 — Station-Filter, Demand-Regime, Train/Val/Test Split
# ══════════════════════════════════════════════════════════════
def prepare_stations_and_splits(df: pd.DataFrame, cfg, city_name=""):
    n_days = (df["hour_ts"].max() - df["hour_ts"].min()).days + 1
    min_events = int(n_days * cfg.min_events_per_day)

    station_volume = df.groupby("station_id")["total_demand"].sum()
    active_ids = station_volume[station_volume >= min_events].index.tolist()
    df = df[df["station_id"].isin(active_ids)].copy()

    station_profile = (
        df.groupby(["station_id", "station_name"], as_index=False)
          .agg(
              avg_total_demand_h=("total_demand", "mean"),
              avg_lends_h=("n_lends", "mean"),
              avg_returns_h=("n_returns", "mean"),
              latitude=("latitude", "first"),
              longitude=("longitude", "first")
          )
    )
    q1 = station_profile["avg_total_demand_h"].quantile(cfg.low_demand_q)
    q2 = station_profile["avg_total_demand_h"].quantile(cfg.high_demand_q)
    station_profile["demand_regime"] = station_profile["avg_total_demand_h"].apply(
        lambda x: "low" if x <= q1 else ("mid" if x <= q2 else "high")
    )
    df = df.merge(
        station_profile[["station_id", "demand_regime", "avg_total_demand_h"]],
        on="station_id", how="left"
    )

    t_min = df["hour_ts"].min()
    t_max = df["hour_ts"].max()
    total_h = (t_max - t_min).total_seconds() / 3600

    train_end = t_min + pd.Timedelta(hours=int(total_h * cfg.train_ratio))
    val_end   = t_min + pd.Timedelta(hours=int(total_h * cfg.val_ratio))

    cn = f" ({city_name})" if city_name else ""
    print(f"  Aktive Stationen{cn}: {df['station_id'].nunique()}")
    print(f"  Regime: {station_profile['demand_regime'].value_counts().to_dict()}")
    print(f"  Zeitraum: {t_min.date()} bis {t_max.date()} ({n_days} Tage)")
    print(f"  Split: Train < {train_end.date()}, Val < {val_end.date()}, Test ab {val_end.date()}")

    return df, station_profile, train_end, val_end

In [28]:
# ══════════════════════════════════════════════════════════════
# 4 — Rolling Context z-Scores + Poisson + ECDF + Labels
# ══════════════════════════════════════════════════════════════
TARGETS = ["n_lends", "n_returns", "net_flow", "total_demand"]
COUNT_TARGETS = ["n_lends", "n_returns", "total_demand"]

def add_context_keys(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()
    x["ctx_hour"] = x["hour_ts"].dt.hour
    x["ctx_dow"]  = x["hour_ts"].dt.dayofweek
    return x

def rolling_context_scores_vectorized(
    full_df: pd.DataFrame, target: str,
    rolling_days: int = 56, min_obs: int = 20
) -> pd.DataFrame:
    x = full_df.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    mu_col    = f"{target}_mu_ctx_roll"
    sd_col    = f"{target}_sd_ctx_roll"
    score_col = f"{target}_z_ctx_roll"

    n_slots = max(rolling_days // 7, 4)
    main_window = n_slots
    main_minp   = min(min_obs, main_window)

    grouped = x.groupby(["station_id", "ctx_hour", "ctx_dow"])[target]
    rolling_mean = grouped.transform(
        lambda s: s.shift(1).rolling(window=main_window, min_periods=main_minp).mean()
    )
    rolling_std = grouped.transform(
        lambda s: s.shift(1).rolling(window=main_window, min_periods=main_minp).std(ddof=0)
    )

    fb1_window = n_slots * 2
    fb1_minp   = min(min_obs, fb1_window)
    grouped_sh = x.groupby(["station_id", "ctx_hour"])[target]
    fb1_mean = grouped_sh.transform(
        lambda s: s.shift(1).rolling(window=fb1_window, min_periods=fb1_minp).mean()
    )
    fb1_std = grouped_sh.transform(
        lambda s: s.shift(1).rolling(window=fb1_window, min_periods=fb1_minp).std(ddof=0)
    )

    fb2_window = n_slots * 4
    fb2_minp   = min(min_obs, fb2_window)
    grouped_s = x.groupby(["station_id"])[target]
    fb2_mean = grouped_s.transform(
        lambda s: s.shift(1).rolling(window=fb2_window, min_periods=fb2_minp).mean()
    )
    fb2_std = grouped_s.transform(
        lambda s: s.shift(1).rolling(window=fb2_window, min_periods=fb2_minp).std(ddof=0)
    )

    mu = rolling_mean.copy()
    sd = rolling_std.copy()
    mask1 = mu.isna()
    mu = mu.where(~mask1, fb1_mean)
    sd = sd.where(~mask1, fb1_std)
    mask2 = mu.isna()
    mu = mu.where(~mask2, fb2_mean)
    sd = sd.where(~mask2, fb2_std)
    sd = sd.clip(lower=1e-6)
    z = (x[target] - mu) / sd

    x[mu_col]    = mu
    x[sd_col]    = sd
    x[score_col] = z
    return x


def add_rolling_poisson_scores_vectorized(
    full_df: pd.DataFrame, target: str,
    rolling_days: int = 56, min_obs: int = 20
) -> pd.DataFrame:
    x = full_df.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    lam_col       = f"{target}_lambda_ctx_roll"
    score_col     = f"{target}_poisson_upper_score"
    score_low_col = f"{target}_poisson_lower_score"
    mu_col        = f"{target}_mu_ctx_roll"

    if mu_col not in x.columns:
        raise ValueError(f"{mu_col} muss zuerst berechnet werden")

    x[lam_col] = x[mu_col].clip(lower=1e-6)
    vals = x[target].values
    lams = x[lam_col].values

    with np.errstate(divide='ignore', invalid='ignore'):
        tail_p = poisson.sf(vals.astype(float) - 1, lams.astype(float))
        score  = -np.log10(np.clip(tail_p, 1e-12, None))
        lower_p   = poisson.cdf(vals.astype(float), lams.astype(float))
        score_low = -np.log10(np.clip(lower_p, 1e-12, None))

    mask_nan = np.isnan(lams)
    score[mask_nan]     = np.nan
    score_low[mask_nan] = np.nan

    x[score_col]     = score
    x[score_low_col] = score_low
    return x


def percentile_score(train_vals: np.ndarray, values: np.ndarray) -> np.ndarray:
    train_vals = np.asarray(train_vals, dtype=float)
    values     = np.asarray(values, dtype=float)
    train_vals = train_vals[np.isfinite(train_vals)]
    if len(train_vals) == 0:
        return np.full(len(values), np.nan, dtype=float)
    train_sorted = np.sort(train_vals)
    ranks = np.searchsorted(train_sorted, values, side="right")
    return ranks / len(train_sorted)


def run_statistical_pipeline(df, cfg, train_end):
    df = add_context_keys(df)
    for t in TARGETS:
        df = rolling_context_scores_vectorized(df, t, cfg.rolling_days, cfg.min_context_obs)
    for t in COUNT_TARGETS:
        df = add_rolling_poisson_scores_vectorized(df, t, cfg.rolling_days, cfg.min_context_obs)

    train_mask = df["hour_ts"] < train_end
    score_cols = [c for c in df.columns if c.endswith("_z_ctx_roll")]

    for c in score_cols:
        train_vals = df.loc[train_mask, c].dropna().values
        df[f"{c}_ecdf"] = percentile_score(train_vals, df[c].values)

    def label_row(row):
        for t in TARGETS:
            z_col = f"{t}_z_ctx_roll"
            if z_col in row.index and pd.notna(row[z_col]):
                if row[z_col] >= 3.0:
                    return "anomal_high"
                if row[z_col] <= -3.0:
                    return "anomal_low"
        return "normal"

    df["label_eval"] = df.apply(label_row, axis=1)
    return df

In [29]:
# ══════════════════════════════════════════════════════════════
# 5 — Sequenzbuilder (Window-Level Labels)
# ══════════════════════════════════════════════════════════════
def make_sequences_with_window_labels(
    x: pd.DataFrame, feature_cols: List[str], window_size: int,
    synth_label_col: str = "synth_label",
    synth_type_col: str = "synth_type",
    require_contiguous: bool = True,
    agg_minutes: int = 60
) -> Tuple[np.ndarray, pd.DataFrame]:
    X_parts, meta_parts = [], []
    expected_ns = pd.Timedelta(minutes=agg_minutes).value

    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").reset_index(drop=True)
        n = len(g)
        if n < window_size:
            continue

        vals = g[feature_cols].to_numpy(dtype=np.float32)
        win  = sliding_window_view(vals, window_shape=window_size, axis=0)
        win  = np.moveaxis(win, -1, 1)

        valid_mask = np.ones(n - window_size + 1, dtype=bool)

        if require_contiguous:
            ts_int = pd.to_datetime(g["hour_ts"]).astype("int64").to_numpy()
            diffs  = np.diff(ts_int)
            step_ok = (diffs == expected_ns).astype(np.int8)
            if window_size > 1:
                ok_sums = np.convolve(step_ok, np.ones(window_size-1, dtype=np.int32), mode="valid")
                valid_mask = (ok_sums == (window_size - 1))

        if not valid_mask.any():
            continue

        end_indices = np.arange(window_size - 1, n)[valid_mask]
        X_parts.append(win[valid_mask])

        meta_chunk = g.iloc[end_indices].copy()

        synth_arr = g[synth_label_col].to_numpy()
        type_arr  = g[synth_type_col].to_numpy()

        window_labels, window_types, window_counts = [], [], []
        for end_idx in end_indices:
            start_idx = end_idx - window_size + 1
            window_synth = synth_arr[start_idx:end_idx + 1]
            window_type  = type_arr[start_idx:end_idx + 1]

            has_synth = int(window_synth.max())
            n_synth   = int(window_synth.sum())

            if has_synth:
                synth_positions = np.where(window_synth == 1)[0]
                wtype = window_type[synth_positions[-1]]
            else:
                wtype = "none"

            window_labels.append(has_synth)
            window_types.append(wtype)
            window_counts.append(n_synth)

        meta_chunk["window_synth_label"] = window_labels
        meta_chunk["window_synth_type"]  = window_types
        meta_chunk["n_synth_in_window"]  = window_counts
        meta_parts.append(meta_chunk)

    if X_parts:
        X    = np.concatenate(X_parts, axis=0)
        meta = pd.concat(meta_parts, axis=0).reset_index(drop=True)
    else:
        X    = np.empty((0, window_size, len(feature_cols)), dtype=np.float32)
        meta = pd.DataFrame()

    return X, meta

In [30]:
# ══════════════════════════════════════════════════════════════
# 6 — Model Builders (AE + Forecaster)
# ══════════════════════════════════════════════════════════════
def build_lstm_autoencoder(
    n_input_features: int,
    window_size: int,
    latent_dim: int = 32,
    n_layers: int = 2,
    dropout: float = 0.1,
    bidirectional: bool = False,
    n_output_features: Optional[int] = None,
) -> keras.Model:
    n_output_features = n_output_features or n_input_features
    inputs = keras.Input(shape=(window_size, n_input_features), name="encoder_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        lstm = layers.LSTM(
            latent_dim, return_sequences=return_sequences,
            dropout=dropout, name=f"encoder_lstm_{i+1}"
        )
        x = layers.Bidirectional(lstm, name=f"encoder_bi_{i+1}")(x) if bidirectional else lstm(x)
    latent = x
    x = layers.RepeatVector(window_size, name="repeat_vector")(latent)
    for i in range(n_layers):
        lstm = layers.LSTM(
            latent_dim, return_sequences=True,
            dropout=dropout, name=f"decoder_lstm_{i+1}"
        )
        x = layers.Bidirectional(lstm, name=f"decoder_bi_{i+1}")(x) if bidirectional else lstm(x)
    outputs = layers.TimeDistributed(
        layers.Dense(n_output_features), name="output_dense"
    )(x)
    return keras.Model(inputs, outputs, name="lstm_autoencoder")


def build_lstm_forecaster(
    n_input_features: int,
    n_output_features: int,
    context_steps: int = 23,
    latent_dim: int = 32,
    n_layers: int = 2,
    dropout: float = 0.1,
) -> keras.Model:
    inputs = keras.Input(shape=(context_steps, n_input_features), name="forecast_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        x = layers.LSTM(
            latent_dim, return_sequences=return_sequences,
            dropout=dropout, name=f"forecast_lstm_{i+1}"
        )(x)
    outputs = layers.Dense(n_output_features, name="forecast_output")(x)
    return keras.Model(inputs, outputs, name="lstm_forecaster")


def train_model_generic(model, X_train, Y_train, X_val, Y_val,
                        epochs, lr, batch_size, early_stop, verbose=1):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss="mse"
    )
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=early_stop,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, verbose=1
        )
    ]
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose
    )
    return model, history

In [31]:
# ══════════════════════════════════════════════════════════════
# 7 — Scoring Helpers (AE + Forecaster)
# ══════════════════════════════════════════════════════════════
def predict_in_batches(model, X, batch_size=2048):
    preds = []
    for i in range(0, len(X), batch_size):
        preds.append(model.predict(X[i:i + batch_size], verbose=0))
    return np.concatenate(preds, axis=0) if preds else np.empty((0, 0, 0), dtype=np.float32)


def compute_reconstruction_scores(X, X_hat, input_features, score_features,
                                  mode="last_step_mean"):
    score_idx = [input_features.index(c) for c in score_features]
    err = (X - X_hat) ** 2
    err_score = err[:, :, score_idx]
    if mode == "last_step_mean":
        return err_score[:, -1, :].mean(axis=1)
    elif mode == "window_mean":
        return err_score.mean(axis=(1, 2))
    elif mode == "max_over_features":
        return err_score[:, -1, :].max(axis=1)
    else:
        raise ValueError(f"Unbekannter mode: {mode}")


def compute_forecast_scores(model, X_input, X_actual_last_step, score_features,
                            ae_features, mode="directional"):
    X_pred = predict_in_batches(model, X_input, batch_size=2048)
    score_idx = [ae_features.index(f) for f in score_features]
    X_actual = X_actual_last_step[:, score_idx]
    err = (X_pred - X_actual) ** 2
    if mode == "mean":
        return err.mean(axis=1)
    elif mode == "max":
        return err.max(axis=1)
    elif mode == "directional":
        abs_err = np.abs(X_pred - X_actual)
        denom = np.maximum(np.maximum(np.abs(X_actual), np.abs(X_pred)), 1e-6)
        rel_err = abs_err / denom
        return 0.5 * abs_err.mean(axis=1) + 0.5 * rel_err.mean(axis=1)
    else:
        raise ValueError(f"Unbekannter mode: {mode}")

In [32]:
# ══════════════════════════════════════════════════════════════
# 8 — Z-Normalisierung (Hour + Hour×Station)
# ══════════════════════════════════════════════════════════════
def znorm_score_by_hour(meta_df, raw_score_col, val_start, test_start,
                        new_col="score_znorm_hour"):
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    mu_per_hour = {}
    std_per_hour = {}
    for h in range(24):
        vals = meta.loc[val_normal & (hour_col == h), raw_score_col].dropna()
        mu_per_hour[h]  = vals.mean() if len(vals) > 0 else 0.0
        std_per_hour[h] = vals.std()  if len(vals) > 1 else 1.0

    hours = hour_col.values
    raw   = meta[raw_score_col].values
    znorm = np.zeros_like(raw)
    for i in range(len(raw)):
        h = hours[i]
        znorm[i] = (raw[i] - mu_per_hour[h]) / max(std_per_hour[h], 1e-8)
    meta[new_col] = znorm
    return meta, mu_per_hour, std_per_hour


def znorm_score_by_hour_station(meta_df, raw_score_col, val_start, test_start,
                                new_col="score_znorm_hour_station"):
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    meta["_hour_tmp"] = hour_col

    stats = (
        meta[val_normal]
        .groupby(["station_id", "_hour_tmp"])[raw_score_col]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    global_stats = (
        meta[val_normal]
        .groupby("_hour_tmp")[raw_score_col]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={"mean": "g_mean", "std": "g_std"})
    )
    stats = stats.merge(global_stats, on="_hour_tmp", how="left")
    stats["use_mean"] = np.where(stats["count"] >= 10, stats["mean"], stats["g_mean"])
    stats["use_std"]  = np.where(stats["count"] >= 10, stats["std"],  stats["g_std"])
    stats["use_std"]  = stats["use_std"].clip(lower=1e-8)

    meta = meta.merge(
        stats[["station_id", "_hour_tmp", "use_mean", "use_std"]],
        on=["station_id", "_hour_tmp"], how="left"
    )
    meta["use_mean"] = meta["use_mean"].fillna(0.0)
    meta["use_std"]  = meta["use_std"].fillna(1.0).clip(lower=1e-8)
    meta[new_col] = (meta[raw_score_col] - meta["use_mean"]) / meta["use_std"]
    meta = meta.drop(columns=["_hour_tmp", "use_mean", "use_std"])
    return meta

In [33]:
# ══════════════════════════════════════════════════════════════
# 9 — Synthetic Injection (V17b: Spike Only)
# ══════════════════════════════════════════════════════════════
V17B_INJECTION_PROBS = {
    "point_spike": 1.0
}
V17B_COLLECTIVE_BLOCK_LEN = (2, 4)

def inject_synthetic_anomalies_v17b(
    df: pd.DataFrame,
    inject_start: pd.Timestamp,
    injection_rate: float = 0.015,
    seed: int = 42,
    verbose: bool = True
) -> pd.DataFrame:
    probs = list(V17B_INJECTION_PROBS.values())
    type_names = list(V17B_INJECTION_PROBS.keys())
    block_min, block_max = V17B_COLLECTIVE_BLOCK_LEN

    rng = np.random.RandomState(seed)
    out = df.copy()

    out["original_n_lends"]   = out["n_lends"].copy()
    out["original_n_returns"] = out["n_returns"].copy()
    out["synth_label"] = 0
    out["synth_type"]  = "none"

    inject_mask = out["hour_ts"] >= inject_start
    inject_idx  = out[inject_mask].index.to_numpy()

    if len(inject_idx) == 0:
        print("  WARNUNG: Keine Daten ab inject_start!")
        return out

    n_inject = max(1, int(len(inject_idx) * injection_rate))
    inject_df = out.loc[inject_idx]
    has_demand = inject_df[inject_df["total_demand"] > 0].index.to_numpy()

    if len(has_demand) < n_inject:
        n_inject = len(has_demand)

    chosen_idx = rng.choice(has_demand, size=n_inject, replace=False)
    types = rng.choice(type_names, size=n_inject, p=probs)

    injected_indices = set()
    requested_counts = {t: 0 for t in type_names}
    injected_counts  = {t: 0 for t in type_names}

    for idx, anom_type in zip(chosen_idx, types):
        requested_counts[anom_type] += 1
        if idx in injected_indices:
            continue

        row = out.loc[idx]

        if anom_type == "point_spike":
            factor = rng.uniform(2.0, 4.0)
            out.loc[idx, "n_lends"]   = int(row["n_lends"] * factor) + rng.randint(1, 4)
            out.loc[idx, "n_returns"] = int(row["n_returns"] * factor) + rng.randint(1, 4)
            out.loc[idx, "synth_label"] = 1
            out.loc[idx, "synth_type"]  = "point_spike"
            injected_indices.add(idx)
            injected_counts["point_spike"] += 1

    out.loc[list(injected_indices), "total_demand"] = (
        out.loc[list(injected_indices), "n_lends"] +
        out.loc[list(injected_indices), "n_returns"]
    )

    if verbose:
        print(f"  Injection: requested={sum(requested_counts.values())}, "
              f"injected={sum(injected_counts.values())}")
        for t in type_names:
            print(f"    {t}: req={requested_counts[t]}, inj={injected_counts[t]}")

    return out

---
## Weather Integration (NEU in V22b)

Wetter-Daten werden auf Stundenbasis aggregiert und der nächstgelegenen Wetterstation pro Netzwerk zugeordnet.

In [34]:
# ══════════════════════════════════════════════════════════════
# 10a — Wetter laden + aufbereiten
# ══════════════════════════════════════════════════════════════

def load_weather_hourly(weather_path: str) -> pd.DataFrame:
    """
    Lädt Wetter-Daten und aggregiert auf Stundenbasis.
    Pro location_id und Stunde: Mittelwert für temp/humidity/wind,
    Summe für Niederschlag.
    """
    print("Lade Wetter-Daten ...")
    weather = pd.read_parquet(weather_path)
    weather["timestamp"] = pd.to_datetime(weather["timestamp"], utc=True)
    weather["hour_ts"] = weather["timestamp"].dt.floor("1h")

    agg_dict = {
        "temperature":  "mean",
        "humidity":     "mean",
        "precipitation":"sum",
        "wind_speed":   "mean",
    }

    weather_h = (
        weather
        .groupby(["location_id", "hour_ts"], as_index=False)
        .agg(agg_dict)
    )

    print(f"  Wetter-Stationen: {weather_h['location_id'].nunique()}")
    print(f"  Zeitraum: {weather_h['hour_ts'].min()} bis {weather_h['hour_ts'].max()}")
    print(f"  Shape (stündlich): {weather_h.shape}")
    return weather_h


def load_weather_station_coords(weather_path: str, geo_path: str) -> pd.DataFrame:
    """
    Ermittelt lat/lon der Wetterstationen.
    Strategie: Wetter location_ids mit Geo-Tabelle joinen.
    Fallback: Mittelpunkt der Stationen in den Wetterdaten abschätzen.
    """
    weather = pd.read_parquet(weather_path)
    weather_locs = weather["location_id"].unique()

    geo = pd.read_parquet(geo_path)
    weather_geo = geo[geo["location_id"].isin(weather_locs)][
        ["location_id", "latitude", "longitude"]
    ].drop_duplicates("location_id")

    print(f"  Wetterstationen mit Geo-Koordinaten: {len(weather_geo)} / {len(weather_locs)}")
    return weather_geo


def find_nearest_weather_station(
    city_stations_df: pd.DataFrame,
    weather_station_coords: pd.DataFrame
) -> int:
    """
    Findet die nächste Wetterstation für ein Netzwerk.
    Berechnet den Netzwerk-Centroid und sucht die nächste Wetterstation.
    """
    centroid_lat = city_stations_df["latitude"].mean()
    centroid_lon = city_stations_df["longitude"].mean()

    ws = weather_station_coords.copy()
    ws["dist"] = np.sqrt(
        (ws["latitude"] - centroid_lat)**2 +
        (ws["longitude"] - centroid_lon)**2
    )
    nearest = ws.loc[ws["dist"].idxmin()]
    return int(nearest["location_id"])


def merge_weather_to_demand(
    df: pd.DataFrame,
    weather_hourly: pd.DataFrame,
    weather_station_id: int,
    city_name: str = ""
) -> pd.DataFrame:
    """
    Merged Wetter-Features einer Wetterstation auf alle Zeilen eines Stadt-DF.
    Alle Stationen einer Stadt bekommen identische Wetterwerte pro Stunde.
    """
    w = weather_hourly[weather_hourly["location_id"] == weather_station_id].copy()
    w = w.drop(columns=["location_id"])

    # hour_ts tz-awareness angleichen
    if df["hour_ts"].dt.tz is None and w["hour_ts"].dt.tz is not None:
        w["hour_ts"] = w["hour_ts"].dt.tz_localize(None)
    elif df["hour_ts"].dt.tz is not None and w["hour_ts"].dt.tz is None:
        w["hour_ts"] = w["hour_ts"].dt.tz_localize("UTC")

    n_before = len(df)
    df = df.merge(w, on="hour_ts", how="left")

    # NaN-Handling: vorwärts/rückwärts füllen (kurze Lücken),
    # dann verbleibende NaNs mit Spalten-Median
    weather_cols = ["temperature", "humidity", "precipitation", "wind_speed"]
    for col in weather_cols:
        if col in df.columns:
            df[col] = df[col].ffill().bfill()
            remaining_na = df[col].isna().sum()
            if remaining_na > 0:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val if pd.notna(median_val) else 0.0)

    matched = df[weather_cols].notna().all(axis=1).sum()
    cn = f" ({city_name})" if city_name else ""
    print(f"  Wetter-Merge{cn}: Station={weather_station_id}, "
          f"matched={matched:,}/{len(df):,}")

    return df

In [35]:
# ══════════════════════════════════════════════════════════════
# 10b — City Data Pipeline (erweitert um Wetter-Merge)
# ══════════════════════════════════════════════════════════════
def prepare_city_data(demand_path: str, geo_path: str, station_names_path: str,
                      cfg, city_name: str,
                      weather_hourly: pd.DataFrame = None,
                      weather_station_coords: pd.DataFrame = None):
    """
    Lädt und bereitet City-Daten auf.
    Wenn weather_hourly und weather_station_coords gegeben sind,
    werden Wetter-Features gemerged.
    Latitude bleibt als Spalte im DF erhalten (für geo Feature-Set).
    """
    print(f"\n{'='*70}")
    print(f"  DATEN-PIPELINE: {city_name}")
    print(f"{'='*70}")

    demand = pd.read_parquet(demand_path)
    geo    = pd.read_parquet(geo_path)
    snames = pd.read_parquet(station_names_path)
    snames = snames.rename(columns={'id': 'station_name_id', 'name': 'station_name'})

    demand = demand.merge(snames[['station_name_id', 'station_name']], on='station_name_id', how='left')
    demand = demand.merge(geo[['location_id', 'latitude', 'longitude']], on='location_id', how='left')
    demand['timestamp'] = pd.to_datetime(demand['timestamp'], utc=True)

    print(f"  Roh: {len(demand):,} Zeilen, {demand['station_id'].nunique()} Stationen")
    print(f"  Zeitraum: {demand['timestamp'].min().date()} bis {demand['timestamp'].max().date()}")

    print("\n  [1] Aggregation...")
    df = aggregate_station_level(demand, minutes=cfg.aggregation_minutes)
    print(f"      Shape: {df.shape}")

    print("  [2] Gap-Fill...")
    n_before = len(df)
    df = fill_missing_time_bins(df, minutes=cfg.aggregation_minutes)
    print(f"      {n_before:,} -> {len(df):,} (+{len(df)-n_before:,})")

    # ── Wetter-Merge (optional) ──
    weather_station_id = None
    if weather_hourly is not None and weather_station_coords is not None:
        print("  [2b] Wetter-Merge...")
        city_stations = df[["station_id", "latitude", "longitude"]].drop_duplicates("station_id")
        weather_station_id = find_nearest_weather_station(city_stations, weather_station_coords)
        df = merge_weather_to_demand(df, weather_hourly, weather_station_id, city_name)

    print("  [3] Stationen, Regime, Splits...")
    df, station_profile, train_end, val_end = prepare_stations_and_splits(df, cfg, city_name)

    print("  [4] Statistische Scoring-Pipeline...")
    df = run_statistical_pipeline(df, cfg, train_end)

    return df, station_profile, train_end, val_end, weather_station_id

In [36]:
# ══════════════════════════════════════════════════════════════
# 11 — V17b Unified Evaluation
# ══════════════════════════════════════════════════════════════
ANOM_TYPES = ["point_spike"]
REGIMES    = ["high", "mid", "low"]

def evaluate_v17b(
    meta: pd.DataFrame,
    score_col: str,
    experiment_name: str,
    val_start: pd.Timestamp,
    test_start: pd.Timestamp,
    verbose: bool = True
) -> Dict:
    meta = meta.copy()
    meta["split_eval"] = np.where(
        meta["hour_ts"] < val_start, "train",
        np.where(meta["hour_ts"] < test_start, "val", "test")
    )

    val_m  = meta[meta["split_eval"] == "val"].copy()
    test_m = meta[meta["split_eval"] == "test"].copy()

    if len(val_m) == 0 or len(test_m) == 0:
        print(f"  [{experiment_name}] WARNUNG: Keine VAL/TEST-Daten!")
        return {}

    results = {"experiment": experiment_name, "n_val": len(val_m), "n_test": len(test_m)}

    n_test_total = len(test_m)
    n_test_anom  = (test_m["synth_label"] == 1).sum()
    results["anomaly_rate_total"] = n_test_anom / n_test_total
    results["n_test_anom"] = int(n_test_anom)

    for atype in ANOM_TYPES:
        n_t = (test_m["synth_type"] == atype).sum()
        results[f"rate_{atype}"] = n_t / n_test_total
        results[f"n_{atype}"] = int(n_t)

    # Threshold via VAL
    y_val = val_m["synth_label"].astype(int).values
    s_val = val_m[score_col].values
    threshold = None
    if len(np.unique(y_val)) > 1:
        prec_v, rec_v, thr_v = precision_recall_curve(y_val, s_val)
        results["val_pr_auc"] = average_precision_score(y_val, s_val)
        if len(thr_v) > 0:
            f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
            best_idx = np.argmax(f1_v)
            threshold = float(thr_v[best_idx])
            results["val_best_f1"] = float(f1_v[best_idx])
            results["val_threshold"] = threshold

    if threshold is None:
        val_norm = val_m[val_m["synth_label"] == 0][score_col]
        threshold = float(val_norm.quantile(0.99)) if len(val_norm) > 0 else float(test_m[score_col].quantile(0.99))
        results["val_threshold"] = threshold
        results["threshold_source"] = "fallback_99pct"

    # TEST Metriken
    y_test = test_m["synth_label"].astype(int).values
    s_test = test_m[score_col].values
    p_test = (s_test >= threshold).astype(int)

    if len(np.unique(y_test)) > 1:
        results["test_pr_auc"]  = average_precision_score(y_test, s_test)
        results["test_roc_auc"] = roc_auc_score(y_test, s_test)
        results["test_f1"]      = f1_score(y_test, p_test, zero_division=0)
        results["test_prec"]    = precision_score(y_test, p_test, zero_division=0)
        results["test_recall"]  = recall_score(y_test, p_test, zero_division=0)

    for atype in ANOM_TYPES:
        type_mask = test_m["synth_type"] == atype
        normal_mask = test_m["synth_label"] == 0
        sub = test_m[type_mask | normal_mask].copy()
        n_anom = int(type_mask.sum())
        if n_anom > 0 and len(sub) > 0:
            y_t = (sub["synth_type"] == atype).astype(int).values
            y_s = sub[score_col].values
            if len(set(y_t)) > 1:
                results[f"pr_{atype}"] = average_precision_score(y_t, y_s)
            detected = (test_m.loc[type_mask, score_col] >= threshold).sum()
            results[f"dr_{atype}"] = detected / n_anom
        else:
            results[f"pr_{atype}"] = None
            results[f"dr_{atype}"] = None

    # Regime × Type
    regime_rows = []
    for atype in ANOM_TYPES:
        for regime in REGIMES:
            rm = test_m["demand_regime"] == regime
            tm = test_m["synth_type"] == atype
            nm = test_m["synth_label"] == 0
            n_a = int((rm & tm).sum())
            n_n = int((rm & nm).sum())
            pr, f1_val, dr = None, None, None
            if n_a > 0 and n_n > 0:
                sub = test_m[(rm & tm) | (rm & nm)]
                y_t = (sub["synth_type"] == atype).astype(int).values
                y_s = sub[score_col].values
                pr = average_precision_score(y_t, y_s) if len(set(y_t)) > 1 else None
                p_sub = (y_s >= threshold).astype(int)
                f1_val = f1_score(y_t, p_sub, zero_division=0)
                detected = (test_m[rm & tm][score_col] >= threshold).sum()
                dr = detected / n_a
            regime_rows.append({
                "type": atype, "regime": regime,
                "n_anom": n_a, "pr_auc": pr, "f1": f1_val, "dr": dr
            })
    results["regime_detail"] = pd.DataFrame(regime_rows)

    if verbose:
        print(f"\n  [{experiment_name}]")
        print(f"    TEST PR-AUC={results.get('test_pr_auc', 'N/A'):.4f}, "
              f"F1={results.get('test_f1', 'N/A'):.4f}, "
              f"Prec={results.get('test_prec', 'N/A'):.4f}, "
              f"Recall={results.get('test_recall', 'N/A'):.4f}")

    return results

---
## Multi-City Setup + Feature-Set Loop

In [37]:
# ══════════════════════════════════════════════════════════════
# MC-1 — City Discovery + Multi-City Config
# ══════════════════════════════════════════════════════════════
import re
from pathlib import Path

def city_to_key(city_name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", city_name.lower()).strip("_")

def discover_city_demand_paths(cleaned_base: str):
    demand_root = Path(cleaned_base) / "demand"
    city_map = {}
    if not demand_root.exists():
        print(f"WARNUNG: demand_root nicht gefunden: {demand_root}")
        return city_map
    for city_dir in sorted(demand_root.iterdir()):
        if not city_dir.is_dir():
            continue
        p = city_dir / "demand_cleaned.parquet"
        if p.exists():
            city_map[city_dir.name] = str(p)
    return city_map

DISCOVERED_CITY_DEMAND = discover_city_demand_paths(CLEANED_BASE)
CITY_DEMAND_ALL = {**DISCOVERED_CITY_DEMAND, **CITY_DEMAND}

print(f"Gefundene Städte: {len(CITY_DEMAND_ALL)}")

# ── Zielstädte ──
TARGET_CITIES = [
    "Dortmund",
    "Duisburg",
    "Essen",
    "Marburg",
    "Winsen (Luhe)",
]
TARGET_CITIES = [c for c in TARGET_CITIES if c in CITY_DEMAND_ALL]
print("\nTarget Cities:", TARGET_CITIES)

Gefundene Städte: 17

Target Cities: ['Dortmund', 'Duisburg', 'Essen', 'Marburg', 'Winsen (Luhe)']


In [38]:
# ══════════════════════════════════════════════════════════════
# MC-2 — Speicherung + Flatten Helpers
# ══════════════════════════════════════════════════════════════
def save_window_level_scores(meta, city_name, model_name, out_dir,
                             score_cols=None, extra_cols=None):
    score_cols = score_cols or []
    extra_cols = extra_cols or []
    base_cols = [
        "station_id", "hour_ts", "split_eval", "label_eval",
        "synth_label", "synth_type", "demand_regime"
    ]
    cols = [c for c in base_cols + score_cols + extra_cols if c in meta.columns]
    out = meta[cols].copy()
    out["city"] = city_name
    out["city_key"] = city_to_key(city_name)
    out["model"] = model_name

    fpath = f"{out_dir}/{city_to_key(city_name)}__{model_name}__window_scores.parquet"
    out.to_parquet(fpath, index=False)
    print(f"  Gespeichert: {fpath} | rows={len(out):,}")
    return fpath

def flatten_result_dict(d: dict, city_name: str, model_name: str):
    row = {}
    for k, v in d.items():
        if isinstance(v, pd.DataFrame):
            continue
        row[k] = v
    row["city"] = city_name
    row["city_key"] = city_to_key(city_name)
    row["model"] = model_name
    return row

def add_station_demand_weight(meta: pd.DataFrame):
    meta = meta.copy()
    demand_col = next((c for c in ["original_n_lends", "n_lends"] if c in meta.columns), None)
    if demand_col is None:
        stat = meta.groupby("station_id").size().rename("station_activity_raw").reset_index()
    else:
        stat = (
            meta.groupby("station_id")[demand_col]
                .mean()
                .rename("station_activity_raw")
                .reset_index()
        )
    stat["station_activity_log1p"] = np.log1p(stat["station_activity_raw"].clip(lower=0))
    q33 = stat["station_activity_raw"].quantile(0.33)
    q67 = stat["station_activity_raw"].quantile(0.67)
    stat["station_activity_group"] = np.where(
        stat["station_activity_raw"] <= q33, "low",
        np.where(stat["station_activity_raw"] >= q67, "high", "medium")
    )
    meta = meta.merge(stat, on="station_id", how="left")
    return meta

In [39]:
# ══════════════════════════════════════════════════════════════
# MC-3a — Sequenz-Erstellung (liest ae_features / score_features aus Global)
# ══════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler

def build_sequences_for_city(
    df_clean, df_inj, train_end, val_end, city_name, scaler_ext=None
):
    """
    Generische Sequenz-Erstellung pro Stadt.
    Liest ae_features und score_features aus dem globalen Scope
    (werden pro Feature-Set gesetzt).
    """
    global ae_features, score_features, context_steps

    df_c = df_clean.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    df_inj_s = df_inj.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)

    df_c["hour_ts"] = pd.to_datetime(df_c["hour_ts"], utc=True)
    df_inj_s["hour_ts"] = pd.to_datetime(df_inj_s["hour_ts"], utc=True)
    train_end = pd.to_datetime(train_end, utc=True)
    val_end   = pd.to_datetime(val_end, utc=True)

    # Prüfe ob alle Features vorhanden
    missing = [f for f in ae_features if f not in df_c.columns]
    if missing:
        print(f"  [{city_name}] WARNUNG: Fehlende Features: {missing}")
        return None

    df_c = df_c.dropna(subset=ae_features)
    df_inj_s = df_inj_s.dropna(subset=ae_features)

    train_mask_sc = df_c["hour_ts"] < train_end
    if scaler_ext is None:
        scaler = StandardScaler()
        scaler.fit(df_c.loc[train_mask_sc, ae_features])
        print(f"  [{city_name}] Scaler gefittet auf {train_mask_sc.sum():,} Train-Zeilen ({len(ae_features)} Features)")
    else:
        scaler = scaler_ext

    df_inj_scaled = df_inj_s.copy()
    df_inj_scaled.loc[:, ae_features] = scaler.transform(df_inj_s[ae_features])

    meta_cols_to_keep = [
        "synth_label", "synth_type", "original_n_lends", "original_n_returns",
        "demand_regime", "label_eval", "station_id", "hour_ts", "n_lends", "n_returns",
    ]
    for col in meta_cols_to_keep:
        if col in df_inj_s.columns:
            df_inj_scaled[col] = df_inj_s[col].values

    X_all, meta_all = make_sequences_with_window_labels(
        df_inj_scaled,
        feature_cols=ae_features,
        window_size=cfg.ae_window_size,
        synth_label_col="synth_label",
        synth_type_col="synth_type",
        require_contiguous=cfg.require_contiguous,
        agg_minutes=cfg.aggregation_minutes
    )

    if "hour_ts" in meta_all.columns:
        meta_all["hour_ts"] = pd.to_datetime(meta_all["hour_ts"], utc=True)

    meta_all["split_eval"] = np.where(
        meta_all["hour_ts"] < train_end, "train",
        np.where(meta_all["hour_ts"] < val_end, "val", "test")
    )

    print(f"  [{city_name}] Sequenzen: {X_all.shape}")
    print(f"  [{city_name}] Split: {meta_all['split_eval'].value_counts().to_dict()}")

    train_normal = (meta_all["split_eval"] == "train") & (meta_all["label_eval"] == "normal")
    val_normal = (
        (meta_all["split_eval"] == "val") &
        (meta_all["synth_label"] == 0) &
        (meta_all["label_eval"] == "normal")
    )
    X_train = X_all[train_normal.values]
    X_val_c = X_all[val_normal.values]

    print(f"  [{city_name}] X_train (normal): {X_train.shape}, X_val (normal): {X_val_c.shape}")

    return {
        "X_all": X_all, "meta_all": meta_all, "scaler": scaler,
        "X_train": X_train, "X_val_clean": X_val_c,
        "train_normal_mask": train_normal, "val_normal_mask": val_normal,
        "train_end": train_end, "val_end": val_end,
    }

In [40]:
# ══════════════════════════════════════════════════════════════
# MC-3b — Oracle Experiment (liest ae_features / score_features aus Global)
# ══════════════════════════════════════════════════════════════
def run_oracle_experiment(city_data, cfg, city_name, models_dir,
                          fc_model_preloaded=None, ae_model_preloaded=None):
    """Train or load both FC and AE, then evaluate. Reads ae_features/score_features from global."""
    global ae_features, score_features, context_steps

    X_train = city_data["X_train"]
    X_val_c = city_data["X_val_clean"]
    X_all   = city_data["X_all"]
    meta    = city_data["meta_all"].copy()
    train_end = city_data["train_end"]
    val_end   = city_data["val_end"]

    score_idx = [ae_features.index(f) for f in score_features]
    results = {}

    city_key = city_to_key(city_name)

    # --- Forecaster ---
    fc_path = f"{models_dir}/{city_key}_oracle_forecaster.keras"
    if fc_model_preloaded is not None:
        fc_model = fc_model_preloaded
        print(f"  [{city_name}] FC vorgeladen")
    elif os.path.exists(fc_path):
        fc_model = keras.models.load_model(fc_path, compile=False)
        print(f"  [{city_name}] FC geladen von {fc_path}")
    else:
        print(f"  [{city_name}] Training Forecaster (Oracle)...")
        fc_model = build_lstm_forecaster(
            n_input_features=len(ae_features),
            n_output_features=len(score_features),
            context_steps=context_steps,
            latent_dim=cfg.ae_latent_dim,
            n_layers=2,
            dropout=cfg.ae_dropout,
        )
        X_fc_train = X_train[:, :context_steps, :]
        Y_fc_train = X_train[:, -1, :][:, score_idx]
        X_fc_val   = X_val_c[:, :context_steps, :]
        Y_fc_val   = X_val_c[:, -1, :][:, score_idx]

        fc_model, _ = train_model_generic(
            fc_model, X_fc_train, Y_fc_train, X_fc_val, Y_fc_val,
            epochs=cfg.ae_epochs, lr=cfg.ae_lr,
            batch_size=cfg.ae_batch_size, early_stop=cfg.ae_early_stop,
            verbose=0
        )
        fc_model.save(fc_path)
        print(f"  [{city_name}] FC gespeichert → {fc_path}")

    X_fc_input = X_all[:, :context_steps, :]
    X_actual_last = X_all[:, -1, :]
    meta["score_fc_raw"] = compute_forecast_scores(
        fc_model, X_fc_input, X_actual_last,
        score_features, ae_features, mode="directional"
    )
    meta = znorm_score_by_hour_station(
        meta, "score_fc_raw", train_end, val_end,
        new_col="score_fc_znorm_hs"
    )
    results["FC_Oracle"] = evaluate_v17b(
        meta, "score_fc_znorm_hs", f"{city_name}_Oracle_FC",
        val_start=train_end, test_start=val_end
    )

    # --- Autoencoder ---
    ae_path = f"{models_dir}/{city_key}_oracle_autoencoder.keras"
    if ae_model_preloaded is not None:
        ae_model = ae_model_preloaded
        print(f"  [{city_name}] AE vorgeladen")
    elif os.path.exists(ae_path):
        ae_model = keras.models.load_model(ae_path, compile=False)
        print(f"  [{city_name}] AE geladen von {ae_path}")
    else:
        print(f"  [{city_name}] Training Autoencoder (Oracle)...")
        ae_model = build_lstm_autoencoder(
            n_input_features=len(ae_features),
            window_size=cfg.ae_window_size,
            latent_dim=cfg.ae_latent_dim,
            n_layers=cfg.ae_layers,
            dropout=cfg.ae_dropout,
            bidirectional=cfg.use_bidirectional,
        )
        ae_model, _ = train_model_generic(
            ae_model, X_train, X_train, X_val_c, X_val_c,
            epochs=cfg.ae_epochs, lr=cfg.ae_lr,
            batch_size=cfg.ae_batch_size, early_stop=cfg.ae_early_stop,
            verbose=0
        )
        ae_model.save(ae_path)
        print(f"  [{city_name}] AE gespeichert → {ae_path}")

    X_hat = predict_in_batches(ae_model, X_all)
    meta["score_ae_raw"] = compute_reconstruction_scores(
        X_all, X_hat, ae_features, score_features, mode="last_step_mean"
    )
    meta = znorm_score_by_hour_station(
        meta, "score_ae_raw", train_end, val_end,
        new_col="score_ae_znorm_hs"
    )
    results["AE_Oracle"] = evaluate_v17b(
        meta, "score_ae_znorm_hs", f"{city_name}_Oracle_AE",
        val_start=train_end, test_start=val_end
    )

    return results, fc_model, ae_model, meta

---
## Schritt 1: Wetter-Daten + Städte laden

In [41]:
# ══════════════════════════════════════════════════════════════
# Wetter-Daten einmalig laden
# ══════════════════════════════════════════════════════════════
weather_hourly = load_weather_hourly(weather_path)
weather_station_coords = load_weather_station_coords(weather_path, geo_path)

print(f"\nWetterstationen mit Koordinaten: {len(weather_station_coords)}")
display(weather_station_coords.head())

Lade Wetter-Daten ...
  Wetter-Stationen: 120
  Zeitraum: 2023-04-01 00:00:00+00:00 bis 2026-02-02 00:00:00+00:00
  Shape (stündlich): (2938138, 6)
  Wetterstationen mit Geo-Koordinaten: 120 / 120

Wetterstationen mit Koordinaten: 120


,location_id,latitude,longitude
2235,292320,49.245,8.537
2236,292325,50.602,8.644
2237,292335,48.433,7.993
4529,292318,50.849,8.774
4530,292394,49.194,7.371


In [42]:
# ══════════════════════════════════════════════════════════════
# Alle Städte laden + Injection (EINMALIG, mit Wetter-Merge)
# ══════════════════════════════════════════════════════════════
# Die Daten werden 1× geladen. Die Feature-Set-Schleife baut dann
# nur jeweils neue Sequenzen mit unterschiedlichen ae_features.

multi_city_frames = {}  # city_name -> {df_clean, df_inj, profile, train_end, val_end, weather_station_id}

for i, city_name in enumerate(TARGET_CITIES, start=1):
    print("\n" + "="*90)
    print(f"[{i}/{len(TARGET_CITIES)}] CITY PIPELINE: {city_name}")
    print("="*90)

    df_city, profile_city, train_end_city, val_end_city, ws_id = prepare_city_data(
        CITY_DEMAND_ALL[city_name], geo_path, station_names_path, cfg, city_name,
        weather_hourly=weather_hourly,
        weather_station_coords=weather_station_coords
    )

    df_city_inj = inject_synthetic_anomalies_v17b(
        df_city,
        inject_start=train_end_city,
        injection_rate=cfg.injection_rate,
        seed=cfg.injection_seed + i,
        verbose=True
    )

    multi_city_frames[city_name] = {
        "df_clean": df_city,
        "df_inj": df_city_inj,
        "profile": profile_city,
        "train_end": train_end_city,
        "val_end": val_end_city,
        "weather_station_id": ws_id,
    }

print("\nFertig. Geladene Städte:", list(multi_city_frames.keys()))

# Quick Check: Wetter-Spalten vorhanden?
sample_city = list(multi_city_frames.keys())[0]
sample_df = multi_city_frames[sample_city]["df_clean"]
weather_cols = ["temperature", "humidity", "precipitation", "wind_speed"]
for col in weather_cols:
    has = col in sample_df.columns
    print(f"  {col}: {'✓' if has else '✗'} in {sample_city}")
print(f"  latitude: {'✓' if 'latitude' in sample_df.columns else '✗'} in {sample_city}")


[1/5] CITY PIPELINE: Dortmund

  DATEN-PIPELINE: Dortmund
  Roh: 334,022 Zeilen, 100 Stationen
  Zeitraum: 2025-04-15 bis 2026-02-02

  [1] Aggregation...
      Shape: (191594, 22)
  [2] Gap-Fill...
      191,594 -> 680,355 (+488,761)
  [2b] Wetter-Merge...
  Wetter-Merge (Dortmund): Station=292324, matched=680,355/680,355
  [3] Stationen, Regime, Splits...
  Aktive Stationen (Dortmund): 78
  Regime: {'low': 26, 'high': 26, 'mid': 26}
  Zeitraum: 2025-04-15 bis 2026-02-02 (294 Tage)
  Split: Train < 2025-10-29, Val < 2025-12-12, Test ab 2025-12-12
  [4] Statistische Scoring-Pipeline...
  Injection: requested=2715, injected=2715
    point_spike: req=2715, inj=2715

[2/5] CITY PIPELINE: Duisburg

  DATEN-PIPELINE: Duisburg
  Roh: 246,265 Zeilen, 58 Stationen
  Zeitraum: 2025-04-15 bis 2026-02-02

  [1] Aggregation...
      Shape: (116486, 22)
  [2] Gap-Fill...
      116,486 -> 357,180 (+240,694)
  [2b] Wetter-Merge...
  Wetter-Merge (Duisburg): Station=292321, matched=357,180/357,180
  

---
## Schritt 2: Feature-Set Loop — Oracle (FC + AE) pro Feature-Set × Stadt

Für jedes Feature-Set werden:
1. Globale `ae_features` / `score_features` gesetzt
2. Sequenzen pro Stadt gebaut
3. Oracle-Modelle trainiert oder geladen
4. Ergebnisse gespeichert

Ergebnisse landen in `results/v22b_feature_ablation/{feature_set}/`

In [43]:
# ══════════════════════════════════════════════════════════════
# FEATURE-SET LOOP — Oracle (FC + AE) für jedes Feature-Set
# ══════════════════════════════════════════════════════════════

all_oracle_results = []   # Sammelt alle Summary-Rows über alle Feature-Sets

for fs_name, fs_config in FEATURE_SETS.items():
    print("\n" + "▓"*100)
    print(f"▓  FEATURE-SET: {fs_name.upper()}")
    print(f"▓  Features ({len(fs_config['ae_features'])}): {fs_config['ae_features']}")
    print("▓" * 100)

    # ── 1) Globale Feature-Listen setzen ──
    ae_features    = fs_config["ae_features"]
    score_features = fs_config["ae_score_features"]
    context_steps  = cfg.ae_window_size - 1

    # ── 2) Ergebnis-Verzeichnisse pro Feature-Set ──
    fs_results_dir = f"{RESULTS_DIR}/{fs_name}"
    fs_models_dir  = f"{fs_results_dir}/models"
    fs_pred_dir    = f"{fs_results_dir}/predictions"
    fs_summary_dir = f"{fs_results_dir}/summary"
    for d in [fs_results_dir, fs_models_dir, fs_pred_dir, fs_summary_dir]:
        os.makedirs(d, exist_ok=True)

    # ── 3) Sequenzen bauen pro Stadt ──
    multi_city_data = {}
    for city_name, frames in multi_city_frames.items():
        print(f"\n--- Sequenzen: {city_name} ({fs_name}) ---")
        city_data = build_sequences_for_city(
            df_clean=frames["df_clean"],
            df_inj=frames["df_inj"],
            train_end=frames["train_end"],
            val_end=frames["val_end"],
            city_name=city_name,
            scaler_ext=None
        )
        if city_data is not None:
            city_data["meta_all"] = add_station_demand_weight(city_data["meta_all"])
            multi_city_data[city_name] = city_data
        else:
            print(f"  [{city_name}] ÜBERSPRUNGEN (fehlende Features)")

    # ── 4) Oracle pro Stadt ──
    oracle_summary_rows = []
    oracle_meta_by_city = {}

    for city_name, city_data in multi_city_data.items():
        print("\n" + "="*90)
        print(f"ORACLE [{fs_name.upper()}]: {city_name}")
        print("="*90)

        # Keras-Session zurücksetzen für sauberes Training
        keras.backend.clear_session()

        oracle_results, fc_model, ae_model, meta_city = run_oracle_experiment(
            city_data, cfg, city_name, fs_models_dir
        )

        meta_city = add_station_demand_weight(meta_city)
        oracle_meta_by_city[city_name] = meta_city.copy()

        for model_key in ["FC_Oracle", "AE_Oracle"]:
            if model_key in oracle_results and oracle_results[model_key]:
                row = flatten_result_dict(oracle_results[model_key], city_name, model_key)
                row["feature_set"] = fs_name
                row["n_features"] = len(ae_features)
                oracle_summary_rows.append(row)

                # Window-level Scores speichern
                score_col_map = {
                    "FC_Oracle": ["score_fc_raw", "score_fc_znorm_hs"],
                    "AE_Oracle": ["score_ae_raw", "score_ae_znorm_hs"],
                }
                if all(c in meta_city.columns for c in score_col_map[model_key]):
                    save_window_level_scores(
                        meta_city, city_name, f"{model_key}_{fs_name}", fs_pred_dir,
                        score_cols=score_col_map[model_key],
                        extra_cols=["station_activity_raw", "station_activity_log1p",
                                    "station_activity_group"]
                    )

    # ── 5) Summary pro Feature-Set ──
    fs_oracle_df = pd.DataFrame(oracle_summary_rows)
    fs_oracle_path = f"{fs_summary_dir}/oracle_summary_{fs_name}.csv"
    fs_oracle_df.to_csv(fs_oracle_path, index=False)
    print(f"\n{'='*70}")
    print(f"ORACLE SUMMARY [{fs_name.upper()}]:")
    print(f"{'='*70}")
    if "test_pr_auc" in fs_oracle_df.columns:
        display(fs_oracle_df[["city", "model", "feature_set", "n_features",
                               "test_pr_auc", "test_f1", "test_prec", "test_recall"
                               ]].sort_values(["model", "test_pr_auc"], ascending=[True, False]))

    all_oracle_results.extend(oracle_summary_rows)

print("\n\n" + "█"*100)
print("█  ALLE FEATURE-SETS ABGESCHLOSSEN")
print("█" * 100)


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
▓  FEATURE-SET: BASELINE
▓  Features (9): ['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

--- Sequenzen: Dortmund (baseline) ---
  [Dortmund] Scaler gefittet auf 367,593 Train-Zeilen (9 Features)
  [Dortmund] Sequenzen: (546844, 24, 9)
  [Dortmund] Split: {'train': 365799, 'test': 98677, 'val': 82368}
  [Dortmund] X_train (normal): (334037, 24, 9), X_val (normal): (74306, 24, 9)

--- Sequenzen: Duisburg (baseline) ---
  [Duisburg] Scaler gefittet auf 200,040 Train-Zeilen (9 Features)
  [Duisburg] Sequenzen: (291869, 24, 9)
  [Duisburg] Split: {'train': 199051, 'test': 50578, 'val': 42240}
  [Duisburg] X_train (normal): (181749, 24, 9), X_val (normal): (38121, 24, 9)

--- Sequenzen: Essen (baseline) ---
  [Essen] Scaler gefi

,city,model,feature_set,n_features,test_pr_auc,test_f1,test_prec,test_recall
9,Winsen (Luhe),AE_Oracle,baseline,9,0.665646,0.669502,0.630609,0.713509
5,Essen,AE_Oracle,baseline,9,0.595439,0.632258,0.539373,0.763789
1,Dortmund,AE_Oracle,baseline,9,0.574796,0.616337,0.523109,0.750000
3,Duisburg,AE_Oracle,baseline,9,0.475529,0.572621,0.636364,0.520486
7,Marburg,AE_Oracle,baseline,9,0.415690,0.471429,0.519289,0.431646
8,Winsen (Luhe),FC_Oracle,baseline,9,0.716045,0.689597,0.641686,0.745240
4,Essen,FC_Oracle,baseline,9,0.605598,0.684802,0.675613,0.694245
0,Dortmund,FC_Oracle,baseline,9,0.602840,0.617414,0.618113,0.616717
2,Duisburg,FC_Oracle,baseline,9,0.524721,0.560129,0.598276,0.526555
6,Marburg,FC_Oracle,baseline,9,0.462775,0.522318,0.551336,0.496203



▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
▓  FEATURE-SET: WEATHER
▓  Features (13): ['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'temperature', 'humidity', 'precipitation', 'wind_speed']
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

--- Sequenzen: Dortmund (weather) ---
  [Dortmund] Scaler gefittet auf 367,593 Train-Zeilen (13 Features)
  [Dortmund] Sequenzen: (546844, 24, 13)
  [Dortmund] Split: {'train': 365799, 'test': 98677, 'val': 82368}
  [Dortmund] X_train (normal): (334037, 24, 13), X_val (normal): (74306, 24, 13)

--- Sequenzen: Duisburg (weather) ---
  [Duisburg] Scaler gefittet auf 200,040 Train-Zeilen (13 Features)
  [Duisburg] Sequenzen: (291869, 24, 13)
  [Duisburg] Split: {'train': 199051, 'test': 50578, 'val': 42240}
  [Duisburg] X_train (normal): (181749, 24, 13), X_val (normal): (38121, 24

,city,model,feature_set,n_features,test_pr_auc,test_f1,test_prec,test_recall
9,Winsen (Luhe),AE_Oracle,weather,13,0.679044,0.665277,0.615741,0.723481
1,Dortmund,AE_Oracle,weather,13,0.599946,0.619152,0.530964,0.742470
5,Essen,AE_Oracle,weather,13,0.575469,0.657950,0.583488,0.754197
3,Duisburg,AE_Oracle,weather,13,0.491696,0.565382,0.563253,0.567527
7,Marburg,AE_Oracle,weather,13,0.428498,0.456079,0.565190,0.382278
8,Winsen (Luhe),FC_Oracle,weather,13,0.714823,0.686591,0.606250,0.791478
4,Essen,FC_Oracle,weather,13,0.605513,0.687611,0.679953,0.695444
0,Dortmund,FC_Oracle,weather,13,0.595735,0.617547,0.622989,0.612199
2,Duisburg,FC_Oracle,weather,13,0.530707,0.571863,0.573171,0.570561
6,Marburg,FC_Oracle,weather,13,0.460543,0.519625,0.551088,0.491561



▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
▓  FEATURE-SET: GEO
▓  Features (10): ['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'latitude']
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

--- Sequenzen: Dortmund (geo) ---
  [Dortmund] Scaler gefittet auf 367,593 Train-Zeilen (10 Features)
  [Dortmund] Sequenzen: (546844, 24, 10)
  [Dortmund] Split: {'train': 365799, 'test': 98677, 'val': 82368}
  [Dortmund] X_train (normal): (334037, 24, 10), X_val (normal): (74306, 24, 10)

--- Sequenzen: Duisburg (geo) ---
  [Duisburg] Scaler gefittet auf 200,040 Train-Zeilen (10 Features)
  [Duisburg] Sequenzen: (291869, 24, 10)
  [Duisburg] Split: {'train': 199051, 'test': 50578, 'val': 42240}
  [Duisburg] X_train (normal): (181749, 24, 10), X_val (normal): (38121, 24, 10)

--- Sequenzen: Essen (geo) ---
  [Essen] Scaler gef

,city,model,feature_set,n_features,test_pr_auc,test_f1,test_prec,test_recall
9,Winsen (Luhe),AE_Oracle,geo,10,0.668709,0.668913,0.624705,0.719855
5,Essen,AE_Oracle,geo,10,0.592194,0.644153,0.555652,0.766187
1,Dortmund,AE_Oracle,geo,10,0.576364,0.619092,0.524882,0.754518
3,Duisburg,AE_Oracle,geo,10,0.471264,0.562893,0.584013,0.543247
7,Marburg,AE_Oracle,geo,10,0.421728,0.476941,0.549862,0.421097
8,Winsen (Luhe),FC_Oracle,geo,10,0.722346,0.690495,0.609148,0.796917
4,Essen,FC_Oracle,geo,10,0.607210,0.691943,0.683841,0.700240
0,Dortmund,FC_Oracle,geo,10,0.574580,0.623183,0.616974,0.629518
2,Duisburg,FC_Oracle,geo,10,0.494461,0.509946,0.630872,0.427921
6,Marburg,FC_Oracle,geo,10,0.460701,0.519016,0.552381,0.489451




████████████████████████████████████████████████████████████████████████████████████████████████████
█  ALLE FEATURE-SETS ABGESCHLOSSEN
████████████████████████████████████████████████████████████████████████████████████████████████████


---
## Gesamtvergleich: Feature-Sets × Städte × Modelle

In [44]:
# ══════════════════════════════════════════════════════════════
# Gesamtvergleich Feature-Set-Ablation
# ══════════════════════════════════════════════════════════════
all_results_df = pd.DataFrame(all_oracle_results)
all_results_path = f"{RESULTS_DIR}/feature_ablation_oracle_summary.csv"
all_results_df.to_csv(all_results_path, index=False)

print("Gesamtvergleich (alle Feature-Sets × Städte × Modelle):")
display_cols = ["city", "model", "feature_set", "n_features",
                "test_pr_auc", "test_f1", "test_prec", "test_recall"]
display_cols = [c for c in display_cols if c in all_results_df.columns]
display(all_results_df[display_cols].sort_values(
    ["city", "model", "feature_set"],
    ascending=[True, True, True]
).reset_index(drop=True))

print(f"\nGespeichert unter: {all_results_path}")

Gesamtvergleich (alle Feature-Sets × Städte × Modelle):


,city,model,feature_set,n_features,test_pr_auc,test_f1,test_prec,test_recall
0,Dortmund,AE_Oracle,baseline,9,0.574796,0.616337,0.523109,0.750000
1,Dortmund,AE_Oracle,geo,10,0.576364,0.619092,0.524882,0.754518
2,Dortmund,AE_Oracle,weather,13,0.599946,0.619152,0.530964,0.742470
3,Dortmund,FC_Oracle,baseline,9,0.602840,0.617414,0.618113,0.616717
4,Dortmund,FC_Oracle,geo,10,0.574580,0.623183,0.616974,0.629518
5,Dortmund,FC_Oracle,weather,13,0.595735,0.617547,0.622989,0.612199
6,Duisburg,AE_Oracle,baseline,9,0.475529,0.572621,0.636364,0.520486
7,Duisburg,AE_Oracle,geo,10,0.471264,0.562893,0.584013,0.543247
8,Duisburg,AE_Oracle,weather,13,0.491696,0.565382,0.563253,0.567527
9,Duisburg,FC_Oracle,baseline,9,0.524721,0.560129,0.598276,0.526555



Gespeichert unter: /content/drive/MyDrive/BA_Colab/data/v22b_feature_ablation/feature_ablation_oracle_summary.csv


In [45]:
# ══════════════════════════════════════════════════════════════
# Pivot: Feature-Set-Effekt pro Stadt und Modell
# ══════════════════════════════════════════════════════════════
if "test_pr_auc" in all_results_df.columns:
    for model_name in ["FC_Oracle", "AE_Oracle"]:
        sub = all_results_df[all_results_df["model"] == model_name]
        if len(sub) == 0:
            continue

        pivot = sub.pivot_table(
            index="city",
            columns="feature_set",
            values="test_pr_auc",
            aggfunc="first"
        )
        # Spalten ordnen: baseline zuerst
        col_order = [c for c in ["baseline", "weather", "geo"] if c in pivot.columns]
        pivot = pivot[col_order]

        # Delta zu Baseline
        if "baseline" in pivot.columns:
            for fs in ["weather", "geo"]:
                if fs in pivot.columns:
                    pivot[f"Δ_{fs}"] = pivot[fs] - pivot["baseline"]

        print(f"\n{'='*70}")
        print(f"PR-AUC Pivot — {model_name}")
        print(f"{'='*70}")
        display(pivot.round(4))

    # Mittelwerte über alle Städte
    print("\n--- Durchschnitt über alle Städte ---")
    avg = all_results_df.groupby(["model", "feature_set"])["test_pr_auc"].mean().unstack("feature_set")
    col_order = [c for c in ["baseline", "weather", "geo"] if c in avg.columns]
    avg = avg[col_order]
    if "baseline" in avg.columns:
        for fs in ["weather", "geo"]:
            if fs in avg.columns:
                avg[f"Δ_{fs}"] = avg[fs] - avg["baseline"]
    display(avg.round(4))


PR-AUC Pivot — FC_Oracle


feature_set,baseline,weather,geo,Δ_weather,Δ_geo
city,,,,,
Dortmund,0.6028,0.5957,0.5746,-0.0071,-0.0283
Duisburg,0.5247,0.5307,0.4945,0.0060,-0.0303
Essen,0.6056,0.6055,0.6072,-0.0001,0.0016
Marburg,0.4628,0.4605,0.4607,-0.0022,-0.0021
Winsen (Luhe),0.7160,0.7148,0.7223,-0.0012,0.0063



PR-AUC Pivot — AE_Oracle


feature_set,baseline,weather,geo,Δ_weather,Δ_geo
city,,,,,
Dortmund,0.5748,0.5999,0.5764,0.0251,0.0016
Duisburg,0.4755,0.4917,0.4713,0.0162,-0.0043
Essen,0.5954,0.5755,0.5922,-0.0200,-0.0032
Marburg,0.4157,0.4285,0.4217,0.0128,0.0060
Winsen (Luhe),0.6656,0.6790,0.6687,0.0134,0.0031



--- Durchschnitt über alle Städte ---


feature_set,baseline,weather,geo,Δ_weather,Δ_geo
model,,,,,
AE_Oracle,0.5454,0.5549,0.5461,0.0095,0.0006
FC_Oracle,0.5824,0.5815,0.5719,-0.0009,-0.0105


---
## MOMENT (nur Baseline Feature-Set)

MOMENT ist ein univariates Foundation Model — es nutzt nur die Demand-Channels
(n_lends, n_returns) und profitiert nicht von Weather/Geo Features.
Daher läuft MOMENT nur 1× als Vergleichspunkt.

In [46]:
# ══════════════════════════════════════════════════════════════
# MOMENT — Setup
# ══════════════════════════════════════════════════════════════
!pip install -q git+https://github.com/moment-timeseries-foundation-model/moment.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [47]:
import torch
from momentfm import MOMENTPipeline

MOMENT_MODEL_NAME = "AutonLab/MOMENT-1-base"
MOMENT_BATCH_SIZE = 64
MOMENT_SCORE_FEATURES = cfg.ae_score_features  # ["n_lends", "n_returns"]

# MOMENT braucht Baseline-Sequenzen
ae_features    = FEATURE_SETS["baseline"]["ae_features"]
score_features = FEATURE_SETS["baseline"]["ae_score_features"]
context_steps  = cfg.ae_window_size - 1

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Model:", MOMENT_MODEL_NAME)
print("Score-Features:", MOMENT_SCORE_FEATURES)

Device: cuda
Model: AutonLab/MOMENT-1-base
Score-Features: ['n_lends', 'n_returns']


In [48]:
# ══════════════════════════════════════════════════════════════
# MOMENT — Model + Scoring Helpers
# ══════════════════════════════════════════════════════════════
def load_moment_model(model_name=MOMENT_MODEL_NAME, device=device):
    model = MOMENTPipeline.from_pretrained(
        model_name,
        model_kwargs={"task_name": "reconstruction"}
    )
    model.init()
    model = model.to(device)
    model.eval()
    return model


def moment_reconstruction_scores(model, X_all_feat, score_feature_indices,
                                  batch_size=MOMENT_BATCH_SIZE, device=device):
    n = X_all_feat.shape[0]
    n_channels = len(score_feature_indices)
    scores_sum = np.zeros(n, dtype=np.float64)

    with torch.no_grad():
        for ch_pos, feat_idx in enumerate(score_feature_indices):
            ch_scores = []
            for i in range(0, n, batch_size):
                xb = X_all_feat[i:i+batch_size, :, feat_idx]
                xb_t = torch.tensor(
                    xb[:, None, :], dtype=torch.float32, device=device
                )
                input_mask = torch.ones(
                    (xb_t.shape[0], xb_t.shape[-1]),
                    dtype=torch.long, device=device
                )
                out = model(x_enc=xb_t, input_mask=input_mask)
                xhat = out.reconstruction.detach().cpu().numpy()[:, 0, :]
                err_last = ((xb - xhat) ** 2)[:, -1]
                ch_scores.append(err_last.astype(np.float64))
            scores_sum += np.concatenate(ch_scores)

    return (scores_sum / n_channels).astype(np.float32)

In [ ]:
# ══════════════════════════════════════════════════════════════
# MOMENT — Fine-Tune Helpers
# ══════════════════════════════════════════════════════════════
from torch.utils.data import DataLoader, TensorDataset

MOMENT_FT_EPOCHS = 5
MOMENT_FT_BATCH_SIZE = 32
MOMENT_FT_LR = 1e-4
MOMENT_FT_HEAD_ONLY = True
MOMENT_FT_MAX_TRAIN_WINDOWS = None


def maybe_subsample(X, max_n=None, seed=42):
    if (max_n is None) or (len(X) <= max_n):
        return X
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X), size=max_n, replace=False)
    return X[idx]


def make_moment_loaders_from_oracle_data(X_train_feat, X_val_feat,
                                          score_feat_idx, batch_size):
    X_train_demand = X_train_feat[:, :, score_feat_idx].astype(np.float32)
    X_val_demand   = X_val_feat[:, :, score_feat_idx].astype(np.float32)
    train_ds = TensorDataset(torch.from_numpy(X_train_demand))
    val_ds   = TensorDataset(torch.from_numpy(X_val_demand))
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    )


def set_trainable_head_only(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True


def fine_tune_moment_reconstruction(
    model, train_loader, val_loader,
    epochs=5, lr=1e-4, head_only=True, save_path=None
):
    if head_only:
        set_trainable_head_only(model)
        print("  Fine-Tune Mode: HEAD ONLY")
    else:
        for p in model.parameters():
            p.requires_grad = True
        print("  Fine-Tune Mode: FULL MODEL")

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion = torch.nn.MSELoss()
    best_val = np.inf
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for (batch_x,) in train_loader:
            batch_x = batch_x.to(device)
            optimizer.zero_grad()
            n_ch = batch_x.shape[-1]
            loss_sum = 0
            for ch in range(n_ch):
                x_ch = batch_x[:, :, ch:ch+1].permute(0, 2, 1)
                mask = torch.ones(x_ch.shape[0], x_ch.shape[-1], dtype=torch.long, device=device)
                out = model(x_enc=x_ch, input_mask=mask)
                xhat = out.reconstruction
                loss_sum += criterion(xhat, x_ch)
            loss = loss_sum / n_ch
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for (batch_x,) in val_loader:
                batch_x = batch_x.to(device)
                n_ch = batch_x.shape[-1]
                loss_sum = 0
                for ch in range(n_ch):
                    x_ch = batch_x[:, :, ch:ch+1].permute(0, 2, 1)
                    mask = torch.ones(x_ch.shape[0], x_ch.shape[-1], dtype=torch.long, device=device)
                    out = model(x_enc=x_ch, input_mask=mask)
                    xhat = out.reconstruction
                    loss_sum += criterion(xhat, x_ch)
                val_losses.append((loss_sum / n_ch).item())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"  Epoch {epoch}/{epochs} — train_loss={train_loss:.6f}, val_loss={val_loss:.6f}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
    if save_path:
        torch.save(model.state_dict(), save_path)
        print(f"  Modell gespeichert → {save_path}")

    model.eval()
    return model, pd.DataFrame(history)

In [50]:
run_finetune = False  # ← auf True setzen um Fine-Tune zu aktivieren

# ══════════════════════════════════════════════════════════════
# MOMENT — Multi-City Loop (Baseline Feature-Set)
# ══════════════════════════════════════════════════════════════

# Baseline-Sequenzen bauen (falls nicht mehr im Speicher)
ae_features    = FEATURE_SETS["baseline"]["ae_features"]
score_features = FEATURE_SETS["baseline"]["ae_score_features"]
context_steps  = cfg.ae_window_size - 1
score_feat_idx = [ae_features.index(f) for f in MOMENT_SCORE_FEATURES]

moment_models_dir = f"{RESULTS_DIR}/baseline/models"
os.makedirs(moment_models_dir, exist_ok=True)

moment_summary_rows = []

for city_name, frames in multi_city_frames.items():
    print("\n" + "="*90)
    print(f"MOMENT LOOP: {city_name}")
    print("="*90)

    # Baseline-Sequenzen bauen
    city_data = build_sequences_for_city(
        df_clean=frames["df_clean"],
        df_inj=frames["df_inj"],
        train_end=frames["train_end"],
        val_end=frames["val_end"],
        city_name=city_name
    )
    if city_data is None:
        continue
    city_data["meta_all"] = add_station_demand_weight(city_data["meta_all"])

    train_end_c = city_data["train_end"]
    val_end_c   = city_data["val_end"]

    # Zero-Shot
    print(f"\n  [{city_name}] MOMENT Zero-Shot")
    zs_model = load_moment_model()
    zs_meta = city_data["meta_all"].copy()
    zs_meta["score_moment_raw"] = moment_reconstruction_scores(
        zs_model, city_data["X_all"], score_feat_idx,
        batch_size=MOMENT_BATCH_SIZE, device=device
    )
    zs_meta = znorm_score_by_hour_station(
        zs_meta, "score_moment_raw", train_end_c, val_end_c,
        new_col="score_moment_znorm_hs"
    )
    zs_res = evaluate_v17b(
        zs_meta, "score_moment_znorm_hs", f"{city_name}_MOMENT_ZeroShot",
        val_start=train_end_c, test_start=val_end_c
    )

    # Fine-Tune
    if run_finetune:
        print(f"\n  [{city_name}] MOMENT Fine-Tune")
        X_train_ft = maybe_subsample(city_data["X_train"], MOMENT_FT_MAX_TRAIN_WINDOWS, seed=42)
        train_loader, val_loader = make_moment_loaders_from_oracle_data(
            X_train_ft, city_data["X_val_clean"], score_feat_idx,
            batch_size=MOMENT_FT_BATCH_SIZE
        )
        ft_model = load_moment_model()
        city_key = city_to_key(city_name)
        ft_model, ft_history = fine_tune_moment_reconstruction(
            ft_model, train_loader, val_loader,
            epochs=MOMENT_FT_EPOCHS, lr=MOMENT_FT_LR,
            head_only=MOMENT_FT_HEAD_ONLY,
            save_path=f"{moment_models_dir}/{city_key}_moment_ft.pt"
        )
        ft_meta = city_data["meta_all"].copy()
        ft_meta["score_moment_ft_raw"] = moment_reconstruction_scores(
            ft_model, city_data["X_all"], score_feat_idx,
            batch_size=MOMENT_BATCH_SIZE, device=device
        )
        ft_meta = znorm_score_by_hour_station(
            ft_meta, "score_moment_ft_raw", train_end_c, val_end_c,
            new_col="score_moment_ft_znorm_hs"
        )
        ft_res = evaluate_v17b(
            ft_meta, "score_moment_ft_znorm_hs", f"{city_name}_MOMENT_FineTune",
            val_start=train_end_c, test_start=val_end_c
        )

    results_to_log = [("MOMENT_ZeroShot", zs_res)]
    if run_finetune:
        results_to_log.append(("MOMENT_FineTune", ft_res))

    for label, res in results_to_log:
        row = flatten_result_dict(res, city_name, label)
        row["feature_set"] = "baseline"
        row["n_features"] = len(ae_features)
        moment_summary_rows.append(row)

moment_summary_df = pd.DataFrame(moment_summary_rows)
moment_path = f"{RESULTS_DIR}/baseline/summary/moment_summary.csv"
os.makedirs(os.path.dirname(moment_path), exist_ok=True)
moment_summary_df.to_csv(moment_path, index=False)
print("\nMOMENT Summary:")
display(moment_summary_df[["city", "model", "test_pr_auc", "test_f1"]].sort_values(
    ["model", "test_pr_auc"], ascending=[True, False]))


MOMENT LOOP: Dortmund
  [Dortmund] Scaler gefittet auf 367,593 Train-Zeilen (9 Features)
  [Dortmund] Sequenzen: (546844, 24, 9)
  [Dortmund] Split: {'train': 365799, 'test': 98677, 'val': 82368}
  [Dortmund] X_train (normal): (334037, 24, 9), X_val (normal): (74306, 24, 9)

  [Dortmund] MOMENT Zero-Shot



  [Dortmund_MOMENT_ZeroShot]
    TEST PR-AUC=0.6131, F1=0.6238, Prec=0.5406, Recall=0.7372

MOMENT LOOP: Duisburg
  [Duisburg] Scaler gefittet auf 200,040 Train-Zeilen (9 Features)
  [Duisburg] Sequenzen: (291869, 24, 9)
  [Duisburg] Split: {'train': 199051, 'test': 50578, 'val': 42240}
  [Duisburg] X_train (normal): (181749, 24, 9), X_val (normal): (38121, 24, 9)

  [Duisburg] MOMENT Zero-Shot

  [Duisburg_MOMENT_ZeroShot]
    TEST PR-AUC=0.5306, F1=0.5735, Prec=0.6054, Recall=0.5448

MOMENT LOOP: Essen
  [Essen] Scaler gefittet auf 282,650 Train-Zeilen (9 Features)
  [Essen] Sequenzen: (363998, 24, 9)
  [Essen] Split: {'train': 243790, 'test': 65296, 'val': 54912}
  [Essen] X_train (normal): (223024, 24, 9), X_val (normal): (49466, 24, 9)

  [Essen] MOMENT Zero-Shot

  [Essen_MOMENT_ZeroShot]
    TEST PR-AUC=0.6780, F1=0.6802, Prec=0.6251, Recall=0.7458

MOMENT LOOP: Marburg
  [Marburg] Scaler gefittet auf 1,002,875 Train-Zeilen (9 Features)
  [Marburg] Sequenzen: (966914, 24, 9)
  

,city,model,test_pr_auc,test_f1
4,Winsen (Luhe),MOMENT_ZeroShot,0.692226,0.675214
2,Essen,MOMENT_ZeroShot,0.678049,0.680153
0,Dortmund,MOMENT_ZeroShot,0.613096,0.623766
1,Duisburg,MOMENT_ZeroShot,0.530615,0.573482
3,Marburg,MOMENT_ZeroShot,0.442464,0.475743


---
## Grand Summary: Oracle (3 Feature-Sets) + MOMENT

In [51]:
# ══════════════════════════════════════════════════════════════
# Grand Summary
# ══════════════════════════════════════════════════════════════
grand_df = pd.concat([
    pd.DataFrame(all_oracle_results),
    moment_summary_df
], ignore_index=True)

grand_path = f"{RESULTS_DIR}/grand_summary_feature_ablation.csv"
grand_df.to_csv(grand_path, index=False)

print("Grand Summary (alle Modelle × Feature-Sets × Städte):")
display_cols = ["city", "model", "feature_set", "n_features",
                "test_pr_auc", "test_f1", "test_prec", "test_recall"]
display_cols = [c for c in display_cols if c in grand_df.columns]
display(grand_df[display_cols].sort_values(
    ["city", "feature_set", "model"],
    ascending=[True, True, True]
).reset_index(drop=True))

print(f"\nGespeichert: {grand_path}")

# Durchschnitt
print("\n--- Durchschnittliche PR-AUC über alle Städte ---")
avg = grand_df.groupby(["model", "feature_set"])["test_pr_auc"].agg(["mean", "std", "count"])
display(avg.round(4))

Grand Summary (alle Modelle × Feature-Sets × Städte):


,city,model,feature_set,n_features,test_pr_auc,test_f1,test_prec,test_recall
0,Dortmund,AE_Oracle,baseline,9,0.574796,0.616337,0.523109,0.750000
1,Dortmund,FC_Oracle,baseline,9,0.602840,0.617414,0.618113,0.616717
2,Dortmund,MOMENT_ZeroShot,baseline,9,0.613096,0.623766,0.540585,0.737199
3,Dortmund,AE_Oracle,geo,10,0.576364,0.619092,0.524882,0.754518
4,Dortmund,FC_Oracle,geo,10,0.574580,0.623183,0.616974,0.629518
5,Dortmund,AE_Oracle,weather,13,0.599946,0.619152,0.530964,0.742470
6,Dortmund,FC_Oracle,weather,13,0.595735,0.617547,0.622989,0.612199
7,Duisburg,AE_Oracle,baseline,9,0.475529,0.572621,0.636364,0.520486
8,Duisburg,FC_Oracle,baseline,9,0.524721,0.560129,0.598276,0.526555
9,Duisburg,MOMENT_ZeroShot,baseline,9,0.530615,0.573482,0.605396,0.544765



Gespeichert: /content/drive/MyDrive/BA_Colab/data/v22b_feature_ablation/grand_summary_feature_ablation.csv

--- Durchschnittliche PR-AUC über alle Städte ---


mean     std  count
model           feature_set                       
AE_Oracle       baseline     0.5454  0.0994      5
                geo          0.5461  0.0989      5
                weather      0.5549  0.0973      5
FC_Oracle       baseline     0.5824  0.0955      5
                geo          0.5719  0.1028      5
                weather      0.5815  0.0946      5
MOMENT_ZeroShot baseline     0.5913  0.1049      5

#Chronos 2

In [59]:
!pip install -q chronos-forecasting

In [62]:
# CHRONOS-2 — Model + Scoring Helpers
# ══════════════════════════════════════════════════════════════
import torch
import numpy as np
from chronos import BaseChronosPipeline

CHRONOS_MODEL_NAME = "amazon/chronos-2"
CHRONOS_BATCH_SIZE = 16   # Chronos-2 ist speicherhungriger als MOMENT
CHRONOS_SCORE_FEATURES = cfg.ae_score_features  # ["n_lends", "n_returns"]

device_map = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Chronos-2 Device: {device_map}")
print(f"Score-Features: {CHRONOS_SCORE_FEATURES}")


def load_chronos2_model(model_name=CHRONOS_MODEL_NAME, device_map=device_map):
    """Lädt Chronos-2 als Forecaster-Pipeline."""
    pipeline = BaseChronosPipeline.from_pretrained(
        model_name,
        device_map=device_map,
    )
    return pipeline


def chronos2_forecast_scores(
    pipeline,
    X_all_feat,
    score_feature_indices,
    context_steps=23,
    batch_size=CHRONOS_BATCH_SIZE,
):
    """
    Chronos-2 Zero-Shot Forecast-Scoring.

    Paradigma (analog zu LSTM-FC):
    - Nimmt die ersten `context_steps` Steps als Context
    - Prognostiziert 1 Step voraus (prediction_length=1)
    - Berechnet Forecast-Error zum tatsächlichen letzten Step
    - Score = Mittlerer Fehler über Score-Features (n_lends, n_returns)

    Nutzt dieselben X_all Fenster wie Oracle (9 Features, 24 Steps).
    Extrahiert nur die Score-Features (n_lends, n_returns) als
    univariate Reihen (pro Channel) — analog zum MOMENT-Scoring.

    Args:
        pipeline: Chronos2Pipeline
        X_all_feat: [N, 24, F] — dieselben Fenster wie Oracle
        score_feature_indices: Indizes von n_lends, n_returns in ae_features
        context_steps: Anzahl Context-Steps (23 = Window - 1)
        batch_size: Batch-Größe

    Returns:
        scores: [N] — Forecast-Error (directional, analog FC)
    """
    n = X_all_feat.shape[0]
    n_channels = len(score_feature_indices)
    scores_sum = np.zeros(n, dtype=np.float64)

    for ch_pos, feat_idx in enumerate(score_feature_indices):
        print(f"    Channel {ch_pos+1}/{n_channels} (feat_idx={feat_idx}) ...")
        ch_scores = []

        for i in range(0, n, batch_size):
            # Context: erste 23 Steps des Demand-Channels
            xb_context = X_all_feat[i:i+batch_size, :context_steps, feat_idx]  # [B, 23]
            # Actual: letzter Step (Step 24)
            xb_actual = X_all_feat[i:i+batch_size, -1, feat_idx]  # [B]

            # Chronos-2 erwartet torch tensor [B, T]
            context_tensor = torch.tensor(xb_context, dtype=torch.float32).unsqueeze(1)  # [B, 1, T]

            # Forecast: 1 Step voraus, Median der Quantile
            with torch.no_grad():
                forecast = pipeline.predict(
                    context_tensor,
                    prediction_length=1,
                )

            # Chronos-2 gibt eine Liste von Tensoren zurück: [tensor([B, num_samples, 1]), ...]
            # Bei univariatem Input: Liste mit 1 Element
            if isinstance(forecast, list):
                forecast = forecast[0]  # [B, num_samples, 1]

            # Median über Samples als Punkt-Prognose
            forecast_median = forecast.median(dim=1).values.cpu().numpy()[:, 0]  # [B]

            # Directional Scoring (identisch zu LSTM-FC)
            abs_err = np.abs(forecast_median - xb_actual)
            denom = np.maximum(
                np.maximum(np.abs(xb_actual), np.abs(forecast_median)),
                1e-6
            )
            rel_err = abs_err / denom
            batch_score = 0.5 * abs_err + 0.5 * rel_err

            ch_scores.append(batch_score.astype(np.float64))

        scores_sum += np.concatenate(ch_scores)

    result = (scores_sum / n_channels).astype(np.float32)
    # NaN-Handling: NaN → 0.0 (neutraler Score, "nicht anomal")
    nan_count = np.isnan(result).sum()
    if nan_count > 0:
        print(f"    WARNUNG: {nan_count} NaN-Scores → auf 0.0 gesetzt")
        result = np.nan_to_num(result, nan=0.0)
    return result

Chronos-2 Device: cuda
Score-Features: ['n_lends', 'n_returns']


In [ ]:
# CHRONOS-2 — Multi-City Zero-Shot Loop (Baseline Feature-Set)
# ══════════════════════════════════════════════════════════════
# Chronos-2 nutzt dieselben Baseline-Sequenzen wie MOMENT.
# Score-Paradigma: Forecast-Error (wie LSTM-FC), nicht Reconstruction.

ae_features    = FEATURE_SETS["baseline"]["ae_features"]
score_features = FEATURE_SETS["baseline"]["ae_score_features"]
context_steps  = cfg.ae_window_size - 1  # 23
score_feat_idx = [ae_features.index(f) for f in CHRONOS_SCORE_FEATURES]

print(f"Score-Features: {CHRONOS_SCORE_FEATURES}")
print(f"Indizes in ae_features: {score_feat_idx}")
print(f"Context Steps: {context_steps}")

# Modell einmalig laden
chronos_pipeline = load_chronos2_model()

chronos_summary_rows = []

for city_name, frames in multi_city_frames.items():
    print("\n" + "="*90)
    print(f"CHRONOS-2 ZERO-SHOT: {city_name}")
    print("="*90)

    # Baseline-Sequenzen bauen
    city_data = build_sequences_for_city(
        df_clean=frames["df_clean"],
        df_inj=frames["df_inj"],
        train_end=frames["train_end"],
        val_end=frames["val_end"],
        city_name=city_name
    )
    if city_data is None:
        print(f"  [{city_name}] ÜBERSPRUNGEN")
        continue

    city_data["meta_all"] = add_station_demand_weight(city_data["meta_all"])
    train_end_c = city_data["train_end"]
    val_end_c   = city_data["val_end"]

    # Zero-Shot Forecast Scoring
    print(f"\n  [{city_name}] Chronos-2 Zero-Shot Scoring ...")
    chronos_meta = city_data["meta_all"].copy()
    chronos_meta["score_chronos2_raw"] = chronos2_forecast_scores(
        chronos_pipeline,
        city_data["X_all"],
        score_feat_idx,
        context_steps=context_steps,
        batch_size=CHRONOS_BATCH_SIZE,
    )

    # Z-Norm (identisch wie bei FC und MOMENT)
    chronos_meta = znorm_score_by_hour_station(
        chronos_meta,
        raw_score_col="score_chronos2_raw",
        val_start=train_end_c,
        test_start=val_end_c,
        new_col="score_chronos2_znorm_hs"
    )

    # Evaluation (identisch wie bei allen anderen Modellen)
    chronos_res = evaluate_v17b(
        chronos_meta,
        score_col="score_chronos2_znorm_hs",
        experiment_name=f"{city_name}_Chronos2_ZeroShot",
        val_start=train_end_c,
        test_start=val_end_c
    )

    row = flatten_result_dict(chronos_res, city_name, "Chronos2_ZeroShot")
    row["feature_set"] = "baseline"
    row["n_features"] = len(ae_features)
    chronos_summary_rows.append(row)

    # Window-Scores speichern
    save_window_level_scores(
        chronos_meta, city_name, "Chronos2_ZeroShot",
        f"{RESULTS_DIR}/baseline/predictions",
        score_cols=["score_chronos2_raw", "score_chronos2_znorm_hs"],
        extra_cols=["station_activity_raw", "station_activity_log1p",
                    "station_activity_group"]
    )

chronos_summary_df = pd.DataFrame(chronos_summary_rows)
chronos_path = f"{RESULTS_DIR}/baseline/summary/chronos2_summary.csv"
os.makedirs(os.path.dirname(chronos_path), exist_ok=True)
chronos_summary_df.to_csv(chronos_path, index=False)

print("\n" + "="*70)
print("CHRONOS-2 ZERO-SHOT SUMMARY:")
print("="*70)
cols = ["city", "model", "test_pr_auc", "test_f1", "test_prec", "test_recall"]
cols = [c for c in cols if c in chronos_summary_df.columns]
display(chronos_summary_df[cols].sort_values("test_pr_auc", ascending=False))


Score-Features: ['n_lends', 'n_returns']
Indizes in ae_features: [0, 1]
Context Steps: 23


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]


CHRONOS-2 ZERO-SHOT: Dortmund
  [Dortmund] Scaler gefittet auf 367,593 Train-Zeilen (9 Features)
  [Dortmund] Sequenzen: (546844, 24, 9)
  [Dortmund] Split: {'train': 365799, 'test': 98677, 'val': 82368}
  [Dortmund] X_train (normal): (334037, 24, 9), X_val (normal): (74306, 24, 9)

  [Dortmund] Chronos-2 Zero-Shot Scoring ...
    Channel 1/2 (feat_idx=0) ...


In [ ]:
# Grand Summary Update: + Chronos-2
# ══════════════════════════════════════════════════════════════
# Falls grand_df schon existiert (aus dem Oracle + MOMENT Block):
try:
    grand_df_updated = pd.concat([
        grand_df,
        chronos_summary_df
    ], ignore_index=True)
except NameError:
    # Falls grand_df nicht existiert, nur Chronos
    grand_df_updated = chronos_summary_df.copy()

grand_updated_path = f"{RESULTS_DIR}/grand_summary_with_chronos2.csv"
grand_df_updated.to_csv(grand_updated_path, index=False)

print("\nGrand Summary (inkl. Chronos-2):")
display_cols = ["city", "model", "feature_set", "test_pr_auc", "test_f1"]
display_cols = [c for c in display_cols if c in grand_df_updated.columns]
display(grand_df_updated[display_cols].sort_values(
    ["city", "test_pr_auc"],
    ascending=[True, False]
).reset_index(drop=True))

# Durchschnitt über alle Städte
print("\n--- Ø PR-AUC über alle Städte (nur Zero-Shot + Oracle) ---")
zs_models = grand_df_updated[
    grand_df_updated["model"].isin([
        "FC_Oracle", "AE_Oracle",
        "MOMENT_ZeroShot", "MOMENT_FineTune",
        "Chronos2_ZeroShot"
    ])
]
avg = zs_models.groupby("model")["test_pr_auc"].agg(["mean", "std", "count"])
display(avg.round(4).sort_values("mean", ascending=False))
